In [5]:
import numpy as np
import pandas as pd
import nolds

In [6]:
PARAMS_PATH = "embedding_params_31_acf_fnn.csv"
DATA_PATH = "dataset_clean.csv"

params = pd.read_csv(PARAMS_PATH)
params = params.loc[:, ~params.columns.str.contains("^Unnamed")]

df_tda = pd.read_csv(DATA_PATH)

display(params.head())
print(params.columns.tolist())

,column,tau,dimension,uzal_cost
0,Lab1_G1_N3,3,4,NaN
1,Lab1_G2_Fн9,3,5,NaN
2,Lab1_G2_Fc3,3,7,NaN
3,Lab1_G1_N1,26,5,NaN
4,Lab1_G1_N2,14,6,NaN


['column', 'tau', 'dimension', 'uzal_cost']


### Подготовка временного ряда к расчёту показателя Ляпунова

Перед вычислением показателя Ляпунова каждый временной ряд приводится к числовому виду, очищается от пропущенных значений и нормализуется.

Нормализация выполняется по формуле:

```python
x = (x - mean) / std
```



In [7]:
def prepare_series(s):
    x = pd.to_numeric(s, errors="coerce").dropna().to_numpy(dtype=float)

    if len(x) < 10:
        return None, "too short"

    std = np.std(x)

    if std == 0 or np.isnan(std):
        return None, "constant or invalid std"

    x = (x - np.mean(x)) / std

    return x, ""

In [8]:
df_tda = pd.read_csv(DATA_PATH)

print("params:", params.shape)
print("df_tda:", df_tda.shape)
print(params.columns.tolist())

params: (31, 4)
df_tda: (1439, 32)
['column', 'tau', 'dimension', 'uzal_cost']


In [9]:
rows = []

for row in params.itertuples(index=False):
    col = row.column
    tau_delay = int(row.tau)
    emb_dim = int(row.dimension)

    result = {
        "column": col,
        "tau": tau_delay,
        "dimension": emb_dim,
        "lyapunov": np.nan,
        "status": "",
    }

    if col not in df_tda.columns:
        result["status"] = "column not found"
        rows.append(result)
        continue

    x, err = prepare_series(df_tda[col])

    if err:
        result["status"] = err
        rows.append(result)
        continue

    try:
        lyapunov = nolds.lyap_r(
            x,
            emb_dim=emb_dim,     
            lag=tau_delay,      
            fit="poly",        
        )

        result["lyapunov"] = lyapunov
        result["status"] = "ok"

    except Exception as e:
        result["status"] = str(e)

    rows.append(result)

lyapunov_results = pd.DataFrame(rows)
lyapunov_results

c:\Users\user\Desktop\Топология_курсовая\.venv\Lib\site-packages\nolds\measures.py:263: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 359
  warnings.warn(msg.format(min_tsep), RuntimeWarning)


,column,tau,dimension,lyapunov,status
0,Lab1_G1_N3,3,4,0.038040,ok
1,Lab1_G2_Fн9,3,5,0.031222,ok
2,Lab1_G2_Fc3,3,7,0.027900,ok
3,Lab1_G1_N1,26,5,0.027112,ok
4,Lab1_G1_N2,14,6,0.013107,ok
5,Lab1_G2_3F2,3,5,0.037517,ok
6,Lab1_G1_T4ср,51,5,0.027902,ok
7,Lab1_G3_T606,24,9,0.022098,ok
8,Lab1_Qtg,15,7,0.011315,ok
9,Lab1_G2_2F2,2,6,0.037481,ok


In [10]:
lyapunov_results["lyapunov_time"] = np.where(
    lyapunov_results["lyapunov"] > 0,
    1 / lyapunov_results["lyapunov"],
    np.inf
)
def interpret_lyapunov(row):
    lam = row["lyapunov"]
    lyap_time = row["lyapunov_time"]

    if pd.isna(lam):
        return "не рассчитано"

    if lam <= 0:
        return "λ ≤ 0: хаотичность не подтверждается"

    if lyap_time <= 100:
        return "λ > 0 и 1/λ ≤ 100: признаки хаотической динамики"

    return "λ > 0, но 1/λ > 100: значение близко к нулю, выраженная хаотичность не подтверждается"


lyapunov_results["lyapunov_type"] = lyapunov_results.apply(
    interpret_lyapunov,
    axis=1
)

lyapunov_results.sort_values("lyapunov", ascending=False)

,column,tau,dimension,lyapunov,status,lyapunov_time,lyapunov_type
26,Lab1_G3_Турбина_ГГ,16,4,0.047179,ok,21.195999,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
0,Lab1_G1_N3,3,4,0.038040,ok,26.288017,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
5,Lab1_G2_3F2,3,5,0.037517,ok,26.654408,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
9,Lab1_G2_2F2,2,6,0.037481,ok,26.679938,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
1,Lab1_G2_Fн9,3,5,0.031222,ok,32.028608,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
19,Lab1_Hpol,2,10,0.029265,ok,34.170894,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
6,Lab1_G1_T4ср,51,5,0.027902,ok,35.839694,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
2,Lab1_G2_Fc3,3,7,0.027900,ok,35.842081,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
23,Lab1_G4_delta_T4PR,28,5,0.027716,ok,36.080787,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики
3,Lab1_G1_N1,26,5,0.027112,ok,36.883631,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики


In [11]:
hurst_rows = []

for col in lyapunov_results["column"]:
    result = {
        "column": col,
        "hurst_rs": np.nan,
        "hurst_status": "",
    }

    if col not in df_tda.columns:
        result["hurst_status"] = "column not found"
        hurst_rows.append(result)
        continue

    x, err = prepare_series(df_tda[col])

    if err:
        result["hurst_status"] = err
        hurst_rows.append(result)
        continue

    try:
        hurst = nolds.hurst_rs(
            x,
            fit="poly",
            corrected=True,
            unbiased=True,
        )

        result["hurst_rs"] = hurst
        result["hurst_status"] = "ok"

    except Exception as e:
        result["hurst_status"] = str(e)

    hurst_rows.append(result)


hurst_results = pd.DataFrame(hurst_rows)

process_report = lyapunov_results.merge(
    hurst_results,
    on="column",
    how="left",
)

display(process_report)

,column,tau,dimension,lyapunov,status,lyapunov_time,lyapunov_type,hurst_rs,hurst_status
0,Lab1_G1_N3,3,4,0.038040,ok,26.288017,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.485665,ok
1,Lab1_G2_Fн9,3,5,0.031222,ok,32.028608,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.601641,ok
2,Lab1_G2_Fc3,3,7,0.027900,ok,35.842081,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.655055,ok
3,Lab1_G1_N1,26,5,0.027112,ok,36.883631,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.896420,ok
4,Lab1_G1_N2,14,6,0.013107,ok,76.294440,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.892538,ok
5,Lab1_G2_3F2,3,5,0.037517,ok,26.654408,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.732238,ok
6,Lab1_G1_T4ср,51,5,0.027902,ok,35.839694,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.885607,ok
7,Lab1_G3_T606,24,9,0.022098,ok,45.253742,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.859927,ok
8,Lab1_Qtg,15,7,0.011315,ok,88.375513,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.817748,ok
9,Lab1_G2_2F2,2,6,0.037481,ok,26.679938,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,0.737444,ok


In [12]:
h = process_report["hurst_rs"]

process_report["hurst_type"] = np.select(
    [
        h.isna(),
        h < 0,
        h <= 0.4,
        (h > 0.4) & (h <= 0.5),
        (h > 0.5) & (h <= 0.6),
        (h > 0.6) & (h <= 1.0),
        h > 1.0,
    ],
    [
        "не рассчитано",
        "H < 0: значение вне ожидаемого диапазона",
        "антиперсистентный процесс",
        "слабая антиперсистентность, возможна хаотическая динамика",
        "слабая персистентность, возможна хаотическая динамика",
        "персистентный процесс",
        "H > 1: значение вне ожидаемого диапазона",
    ],
    default="не определено",
)

process_report["process_type"] = (
    process_report["hurst_type"]
    + " | "
    + process_report["lyapunov_type"]
)

display(
    process_report[
        [
            "column",
            "tau",
            "dimension",
            "hurst_rs",
            "lyapunov",
            "lyapunov_time",
            "hurst_type",
            "lyapunov_type",
            "process_type",
            "status",
            "hurst_status",
        ]
    ].sort_values("process_type")
)

display(
    process_report["process_type"]
    .value_counts()
    .to_frame("count")
)

,column,tau,dimension,hurst_rs,lyapunov,lyapunov_time,hurst_type,lyapunov_type,process_type,status,hurst_status
14,Lab1_PposleNag,2,7,1.024337,0.026547,37.669224,H > 1: значение вне ожидаемого диапазона,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,H > 1: значение вне ожидаемого диапазона | λ >...,ok,ok
29,Lab1_G3_Коксование,13,3,1.362306,0.014705,68.005283,H > 1: значение вне ожидаемого диапазона,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,H > 1: значение вне ожидаемого диапазона | λ >...,ok,ok
30,Lab1_TdoNag,4,4,1.071130,0.006680,149.697867,H > 1: значение вне ожидаемого диапазона,"λ > 0, но 1/λ > 100: значение близко к нулю, в...",H > 1: значение вне ожидаемого диапазона | λ >...,ok,ok
1,Lab1_G2_Fн9,3,5,0.601641,0.031222,32.028608,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok
2,Lab1_G2_Fc3,3,7,0.655055,0.027900,35.842081,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok
3,Lab1_G1_N1,26,5,0.896420,0.027112,36.883631,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok
4,Lab1_G1_N2,14,6,0.892538,0.013107,76.294440,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok
5,Lab1_G2_3F2,3,5,0.732238,0.037517,26.654408,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok
6,Lab1_G1_T4ср,51,5,0.885607,0.027902,35.839694,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok
7,Lab1_G3_T606,24,9,0.859927,0.022098,45.253742,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,ok,ok


,count
process_type,
персистентный процесс | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,19
"персистентный процесс | λ > 0, но 1/λ > 100: значение близко к нулю, выраженная хаотичность не подтверждается",7
H > 1: значение вне ожидаемого диапазона | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,2
"слабая антиперсистентность, возможна хаотическая динамика | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики",1
"слабая персистентность, возможна хаотическая динамика | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики",1
"H > 1: значение вне ожидаемого диапазона | λ > 0, но 1/λ > 100: значение близко к нулю, выраженная хаотичность не подтверждается",1


In [13]:
import ast

factorization_report = pd.read_csv("factorization_report_without_constants.csv")

# members после чтения из csv становится строкой, возвращаем список
def parse_members(value):
    if isinstance(value, list):
        return value

    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except Exception:
            return [value]

    return [value]


factorization_report["members"] = factorization_report["members"].apply(parse_members)

display(factorization_report.head())

,global_class_id,group,representative,class_size,mean_distance_representative,members
0,1,stationary,Lab1_G1_N3,1,0.000000,[Lab1_G1_N3]
1,2,stationary,Lab1_G2_Fн9,12,0.053783,"[Lab1_G2_Fc2, Lab1_G3_V2, Lab1_G3_КВД, Lab1_G3..."
2,3,stationary,Lab1_G2_Fc3,3,0.015828,"[Lab1_G2_Fc3, Lab1_G2_Fв9, Lab1_G2_2F3]"
3,4,stationary,Lab1_G3_Турбина_ГГ,7,0.040979,"[Lab1_G2_3F1, Lab1_G3_Турбина_ГГ, Lab1_G3_ПО_С..."
4,5,difference_stationary_or_borderline,Lab1_G1_N1,1,0.000000,[Lab1_G1_N1]


In [14]:
process_report_with_classes = process_report.merge(
    factorization_report[
        [
            "global_class_id",
            "group",
            "representative",
            "class_size",
            "members",
        ]
    ],
    left_on="column",
    right_on="representative",
    how="left",
)

display(
    process_report_with_classes[
        [
            "global_class_id",
            "group",
            "representative",
            "class_size",
            "members",
            "hurst_rs",
            "lyapunov",
            "lyapunov_time",
            "hurst_type",
            "lyapunov_type",
            "process_type",
        ]
    ].sort_values(["group", "process_type"])
)

,global_class_id,group,representative,class_size,members,hurst_rs,lyapunov,lyapunov_time,hurst_type,lyapunov_type,process_type
3,5,difference_stationary_or_borderline,Lab1_G1_N1,1,[Lab1_G1_N1],0.896420,0.027112,36.883631,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
4,6,difference_stationary_or_borderline,Lab1_G1_N2,2,"[Lab1_G1_N2, Lab1_G4_N2]",0.892538,0.013107,76.294440,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
5,7,difference_stationary_or_borderline,Lab1_G2_3F2,29,"[Lab1_G1_P2, Lab1_Ne, Lab1_PologenieTRK, Lab1_...",0.732238,0.037517,26.654408,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
6,8,difference_stationary_or_borderline,Lab1_G1_T4ср,2,"[Lab1_G1_T4ср, Lab1_G4_T4]",0.885607,0.027902,35.839694,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
7,9,difference_stationary_or_borderline,Lab1_G3_T606,1,[Lab1_G3_T606],0.859927,0.022098,45.253742,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
8,10,difference_stationary_or_borderline,Lab1_Qtg,1,[Lab1_Qtg],0.817748,0.011315,88.375513,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
20,26,dynamic,Lab1_G1_T1,1,[Lab1_G1_T1],0.902912,0.021411,46.705456,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
21,27,dynamic,Lab1_G1_T600,1,[Lab1_G1_T600],0.906647,0.011364,87.994605,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
22,28,dynamic,Lab1_G1_T638,1,[Lab1_G1_T638],0.937976,0.017045,58.666594,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...
23,29,dynamic,Lab1_G4_delta_T4PR,1,[Lab1_G4_delta_T4PR],0.867003,0.027716,36.080787,персистентный процесс,λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...


In [15]:
process_summary = (
    process_report_with_classes
    .groupby("process_type")
    .agg(
        classes_count=("global_class_id", "count"),
        represented_series_count=("class_size", "sum"),
        representatives=("representative", list),
    )
    .reset_index()
    .sort_values("represented_series_count", ascending=False)
)

display(process_summary)

,process_type,classes_count,represented_series_count,representatives
2,персистентный процесс | λ > 0 и 1/λ ≤ 100: при...,19,63,"[Lab1_G2_Fн9, Lab1_G2_Fc3, Lab1_G1_N1, Lab1_G1..."
3,"персистентный процесс | λ > 0, но 1/λ > 100: з...",7,9,"[Lab1_G3_Pc2, Lab1_TC_Ptgdg, Lab1_dPmg, Lab1_G..."
5,"слабая персистентность, возможна хаотическая д...",1,7,[Lab1_G3_Турбина_ГГ]
0,H > 1: значение вне ожидаемого диапазона | λ >...,2,3,"[Lab1_PposleNag, Lab1_G3_Коксование]"
1,H > 1: значение вне ожидаемого диапазона | λ >...,1,1,[Lab1_TdoNag]
4,"слабая антиперсистентность, возможна хаотическ...",1,1,[Lab1_G1_N3]


In [22]:
process_summary_by_group = (
    process_report_with_classes
    .groupby(["group", "process_type"])
    .agg(
        classes_count=("global_class_id", "count"),
        represented_series_count=("class_size", "sum"),
        representatives=("representative", list),
    )
    .reset_index()
    .sort_values(["group", "represented_series_count"], ascending=[True, False])
)


In [23]:
summary_full = process_summary_by_group.copy()

summary_full["representatives_full"] = summary_full["representatives"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)

pd.set_option("display.max_colwidth", None)
display(
    summary_full[
        [
            "group",
            "process_type",
            "classes_count",
            "represented_series_count",
            "representatives_full",
        ]
    ]
)

,group,process_type,classes_count,represented_series_count,representatives_full
0,difference_stationary_or_borderline,персистентный процесс | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,6,36,"Lab1_G1_N1, Lab1_G1_N2, Lab1_G2_3F2, Lab1_G1_T4ср, Lab1_G3_T606, Lab1_Qtg"
1,dynamic,персистентный процесс | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,4,4,"Lab1_G1_T1, Lab1_G1_T600, Lab1_G1_T638, Lab1_G4_delta_T4PR"
2,dynamic,"персистентный процесс | λ > 0, но 1/λ > 100: значение близко к нулю, выраженная хаотичность не подтверждается",2,2,"Lab1_G4_delta_T4, Lab1_TC_Tm"
6,jump,"персистентный процесс | λ > 0, но 1/λ > 100: значение близко к нулю, выраженная хаотичность не подтверждается",3,3,"Lab1_G3_Pc2, Lab1_TC_Ptgdg, Lab1_dPmg"
3,jump,H > 1: значение вне ожидаемого диапазона | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,1,1,Lab1_G3_Коксование
4,jump,"H > 1: значение вне ожидаемого диапазона | λ > 0, но 1/λ > 100: значение близко к нулю, выраженная хаотичность не подтверждается",1,1,Lab1_TdoNag
5,jump,персистентный процесс | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,1,1,Lab1_Hpol
7,stationary,персистентный процесс | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики,2,15,"Lab1_G2_Fн9, Lab1_G2_Fc3"
9,stationary,"слабая персистентность, возможна хаотическая динамика | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики",1,7,Lab1_G3_Турбина_ГГ
8,stationary,"слабая антиперсистентность, возможна хаотическая динамика | λ > 0 и 1/λ ≤ 100: признаки хаотической динамики",1,1,Lab1_G1_N3
